# 02 · Greedy baseline

A **greedy** algorithm makes the best choice it can *right now* and never changes it later.

For scheduling that means:

1. Put the tasks in order, most important first.
2. Take the next task and find its best (team, start day) on the calendar as it is now.
3. Book it for good, then move on to the next task.

It is fast and easy to explain, which is why real planners use it to build their first schedule. Its weakness is step 3: an early booking can take the team a later, more important fit needed. Notebook 03 will fix that.

In this notebook you write the two decisions (**YOUR TURN** cells). The rest is ready to run.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research').is_dir())
sys.path.insert(0, str(ROOT))

import random
import time
from datetime import timedelta
from itertools import combinations

import matplotlib.pyplot as plt
import pandas as pd

from research.contract import engine_result, scheduled_item, unscheduled_item, work_dates
from research.evaluate import TEAM_SIZE, score, summary, validate
from research.reference import greedy as reference
from research.schedule_tools import Calendar, candidate_score, score_notes
from research.synthetic import Scenario, generate

data = generate(seed=1)

## Your toolbox

- `Calendar(data)` knows which days each technician is taken. `calendar.free_technicians(dates)` lists who is free on all of `dates`; `calendar.book(team, dates, location)` takes those days.
- `work_dates(start, n)` gives `n` consecutive days from `start`.
- `candidate_score(task, team, start, data, calendar)` returns `(points, parts)`: what that placement is worth, using the same rules as the grader.

Try them on one task:

In [ ]:
calendar = Calendar(data)
task = data['tasks'][0]
start = task['earliest_start']
dates = work_dates(start, task['remaining_days'])
free = calendar.free_technicians(dates)

print(task['doc_no'], 'needs', task['remaining_days'], 'day(s) from', start)
print('Free on all those days:', [tech['name'] for tech in free])

team = free[:TEAM_SIZE]
points, parts = candidate_score(task, team, start, data, calendar)
print('With', [tech['name'] for tech in team], '->', round(points, 1), 'points')
dict(parts)

## Decision 1: the order

Whatever goes first gets the best teams and the earliest days, so the order *is* the priority policy. SSJ's rule:

1. Lower `priority_rank` first (0 = XPRJ, 1 = urgent, 2 = normal).
2. Then continuations, so a job already in progress doesn't stop halfway.
3. Then the documents that have waited longest (`initial_lapse_days`, largest first).

In [ ]:
# YOUR TURN
def order_tasks(tasks):
    """Return the tasks in the order the greedy should place them."""
    return sorted(
        tasks,
        key=lambda task: (
            task['priority_rank'],            # 1. XPRJ (0), then urgent (1), then normal (2)
            not task['is_continuation'],      # 2. jobs already in progress first, the not command to reverse the sorted() function
            -task['initial_lapse_days'],      # 3. longest waiting first
            task['id'],                       # 4. tie-breaker, so the order is always the same
        ),
    )


In [ ]:
# Check: compare with the reference. Tasks that tie on all three rules may swap places; that's fine.
mine = order_tasks(data['tasks'])
expected = reference.order_tasks(data['tasks'])
same = [t['id'] for t in mine] == [t['id'] for t in expected]
print('Same order as the reference.' if same else 'Different order: compare the columns below.')
pd.DataFrame({
    'yours': [f"{t['doc_no']} (rank {t['priority_rank']}, waited {t['initial_lapse_days']})" for t in mine],
    'reference': [f"{t['doc_no']} (rank {t['priority_rank']}, waited {t['initial_lapse_days']})" for t in expected],
})

## Decision 2: the best placement for one task

Walk forward day by day from the task's `earliest_start`:

- On each day, find who is free for the whole job. Skip the day if fewer than 2 are.
- Remember the **first day** that had a free team. Keep looking for `lookahead_days` more days after it, then stop.
- On every day you look at, score **every possible pair** of free technicians and keep the best.

Why look past the first free day at all? Starting later costs points (the wait penalty), so a later day only wins when it is clearly better, for example when the first free day would mean working on a Sunday.

In [ ]:
LOOKAHEAD_DAYS = 7
MAX_SEARCH_DAYS = 365


# YOUR TURN
def best_placement(task, data, calendar, lookahead_days=LOOKAHEAD_DAYS, max_search_days=MAX_SEARCH_DAYS):
    """The best (points, team, start, parts) for one task, or None if no team is ever free."""
    best = None          # the best placement found so far
    first_free = None    # the first day any team was free

    for offset in range(max_search_days):
        start = task['earliest_start'] + timedelta(days=offset)

        # Stop once we've looked lookahead_days past the first free day.
        if first_free is not None and (start - first_free).days > lookahead_days:
            break

        dates = work_dates(start, task['remaining_days'])
        free = calendar.free_technicians(dates)
        if len(free) < TEAM_SIZE:
            continue                     # not enough people free on this day: try the next one

        if first_free is None:
            first_free = start           # the first day that works

        for team in combinations(free, TEAM_SIZE):
            points, parts = candidate_score(task, list(team), start, data, calendar)
            if best is None or points > best[0]:
                best = (points, list(team), start, parts)

    return best


In [ ]:
# Check: your best placement vs the reference, for the first five tasks on the same calendar.
calendar = Calendar(data)
rows = []
for task in reference.order_tasks(data['tasks'])[:5]:
    mine = best_placement(task, data, calendar)
    ref = reference.best_placement(task, data, calendar)
    rows.append({
        'task': task['doc_no'],
        'your points': round(mine[0], 1) if mine else None,
        'reference points': round(ref[0], 1) if ref else None,
        'your start': mine[2] if mine else None,
        'reference start': ref[2] if ref else None,
    })
pd.DataFrame(rows)

## The greedy loop

Your two decisions, put together. Same arguments and result as the app's `run_engine()`.

In [ ]:
def my_greedy(tasks, technicians, start_date, holidays, history_ctx, existing_jobs=()):
    started = time.perf_counter()
    data = {'tasks': tasks, 'technicians': technicians, 'start_date': start_date,
            'holidays': holidays, 'history_ctx': history_ctx, 'existing_jobs': existing_jobs}
    calendar = Calendar(data)
    scheduled, unscheduled = [], []
    for task in order_tasks(tasks): # take tasks in order
        found = best_placement(task, data, calendar) # find best slot to put tasks
        if found is None:
            unscheduled.append(unscheduled_item(task, 'No free team was found.'))
            continue
        points, team, start, parts = found[:4]
        calendar.book(team, work_dates(start, task['remaining_days']), task['location']) # book the task
        scheduled.append(scheduled_item(task, team, start, points, score_notes(parts))) # save it into SSJ format
    return engine_result(scheduled, unscheduled, 'my_greedy', started)


result = my_greedy(**data)
print('Problems:', validate(result, data) or 'none')
print('Score:', score(result, data).total)
summary(result, data)

In [ ]:
pd.DataFrame([{
    'doc_no': item['doc_no'],
    'team': ', '.join(member['name'] for member in item['team']),
    'start': item['start_date'],
    'days': len(item['work_dates']),
    'points': item['score'],
    'why': '; '.join(item['score_notes'][:3]),
} for item in result['scheduled']])

## The schedule on a calendar

Grey is work that was already booked; colours are the new suggestions (blue SVC, orange XPRJ, green PRJ). Shaded columns are Sundays, red lines are holidays.

In [ ]:
def draw_schedule(result, data, days=21):
    start = data['start_date']
    row = {tech['id']: n for n, tech in enumerate(data['technicians'])}
    colors = {'svc': '#3b82f6', 'xprj': '#f59e0b', 'prj': '#10b981'}
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(row) + 1))
    for job in data['existing_jobs']:
        ax.barh(row[job['tech_id']], job['days_duration'], left=(job['start_date'] - start).days, color='#cbd5e1')
    for item in result['scheduled']:
        offset = (pd.Timestamp(item['start_date']).date() - start).days
        for member in item['team']:
            ax.barh(row[member['id']], len(item['work_dates']), left=offset,
                    color=colors[item['doc_type']], edgecolor='white')
            ax.text(offset + 0.1, row[member['id']], item['doc_no'].split('/')[0], va='center', fontsize=7, color='white')
    for day in range(-5, days):
        if (start + timedelta(days=day)).weekday() == 6:
            ax.axvspan(day, day + 1, color='#fde68a', alpha=0.4)
    for holiday in data['holidays']:
        ax.axvline((holiday - start).days + 0.5, color='#dc2626')
    ax.set_yticks(list(row.values()))
    ax.set_yticklabels([tech['name'] for tech in data['technicians']])
    ax.set_xlim(-2, days)
    ax.set_xlabel(f'days from {start}')
    ax.invert_yaxis()
    plt.show()


draw_schedule(result, data)

## Twenty problems, side by side

One problem can be lucky. This runs your greedy and the reference on 20 different seeds.

The points shown per task in the table above were worked out *when the task was booked*. The grader scores the finished schedule, where a task booked later can still add a proximity bonus, so totals can differ slightly from adding up the table.

In [ ]:
rows = []
for seed in range(20):
    problem = generate(seed=seed)
    mine, ref = my_greedy(**problem), reference.run_greedy(**problem)
    rows.append({
        'seed': seed,
        'valid': not validate(mine, problem),
        'your score': score(mine, problem).total,
        'reference score': score(ref, problem).total,
        'unscheduled': len(mine['unscheduled']),
        'wait days': summary(mine, problem)['total_wait_days'],
        'ms': mine['runtime_ms'],
    })
comparison = pd.DataFrame(rows)
comparison

## Experiments

Change something, re-run the cells below it, and note what happens. Write your observations in a new markdown cell; they're your research notes.

1. **Lookahead.** Set `LOOKAHEAD_DAYS` to 0, 3 and 14. What happens to the score, the waiting days and the runtime?
2. **Order.** Replace `order_tasks` in `my_greedy` with a random order (`random.sample(tasks, len(tasks))`). How much worse is it on the 20 seeds?
3. **Pressure.** Try `generate(seed=1, scenario=Scenario(n_tasks=40))`. Which tasks wait longest, and which earlier booking took their team?
4. **The weakness.** Find one booking you would change by hand to raise the score. Greedy can never do that, because it never goes back. That's exactly what local search in notebook 03 does.